# RAPTOR on BookSum

End-to-end demo: load a BookSum chapter → build RAPTOR tree → generate summary or answer a question → evaluate against the reference summary.

**Why BookSum?**  
Chapters are long enough that a plain LLM call must truncate, making RAPTOR's hierarchical retrieval genuinely advantageous.

## Section 0 — Setup

In [ ]:
import os, sys, importlib
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.path.abspath('.')), '..', '.env'))
load_dotenv()  # fallback

_root = os.path.abspath('..')
sys.path.insert(0, os.path.join(_root, 'lib'))        # for `from raptor import ...`
sys.path.insert(0, os.path.join(_root, 'pipelines'))  # for pipeline modules

print('OPENAI_API_KEY set:', bool(os.environ.get('OPENAI_API_KEY')))

## Section 1 — Load BookSum chapter

We filter to single chapters only (`is_aggregate=False`) so each row is one chapter with a matching reference summary.

In [2]:
import booksum_pipeline
importlib.reload(booksum_pipeline)
from booksum_pipeline import load_booksum, extract_booksum_text

dataset = load_booksum(split='validation')
print(f'Chapters available: {len(dataset)}')

row = dataset[0]
text, reference, title = extract_booksum_text(row)

print(f'Title     : {title}')
print(f'Words     : {len(text.split())}')
print(f'Ref words : {len(reference.split())}')
print(f'\nChapter preview:\n{text[:400]}...')

c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Filter: 100%|██████████| 1484/1484 [00:00<00:00, 11116.20 examples/s]

Chapters available: 1350
Title     : Bleak House.chapter 18
Words     : 7056
Ref words : 243

Chapter preview:
Lady Dedlock


It was not so easy as it had appeared at first to arrange for
Richard's making a trial of Mr. Kenge's office. Richard himself was
the chief impediment. As soon as he had it in his power to leave Mr.
Badger at any moment, he began to doubt whether he wanted to leave
him at all. He didn't know, he said, really. It wasn't a bad
profession; he couldn't assert that he disliked it; perhap...


## Section 2 — Configure RAPTOR

In [3]:
import qasper_pipeline
importlib.reload(qasper_pipeline)
from qasper_pipeline import build_raptor_config

config = build_raptor_config()
print('Config ready.')

2026-04-07 14:09:15,565 - Use pytorch device_name: cpu
2026-04-07 14:09:15,566 - Load pretrained SentenceTransformer: sentence-transformers/multi-qa-mpnet-base-cos-v1
2026-04-07 14:09:15,715 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/multi-qa-mpnet-base-cos-v1/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-07 14:09:15,729 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/multi-qa-mpnet-base-cos-v1/d51b22a1dfa8184e9258074e56e2875e50612dca/modules.json "HTTP/1.1 200 OK"
2026-04-07 14:09:15,839 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/multi-qa-mpnet-base-cos-v1/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-07 14:09:15,855 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/multi-qa-mpnet-base-cos-v1/d51b22a1dfa8184e9258074e56e2875e50612dca/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-0

Config ready.


## Section 3 — Summarize chapter

In [5]:
importlib.reload(booksum_pipeline)
from booksum_pipeline import summarize_chapter

result = summarize_chapter(row, config=config)
print('SUMMARY:')
print(result['summary'])

2026-04-07 14:11:22,083 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
            Selection Mode: top_k
            Summarization Length: 100
            Summarization Model: <raptor.SummarizationModels.GPT4NanoSummarizationModel object at 0x00000190AB07BCB0>
            Embedding Models: {'EMB': <raptor.EmbeddingModels.SBertEmbeddingModel object at 0x00000190AB07AA50>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        
2026-04-07 14:11:22,084 - Successfully initialized ClusterTreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
           

SUMMARY:
The document presents a series of detailed observations and reflections centered around a serene estate and its social environment. It describes idyllic landscapes, including lush gardens, picturesque water features, and expansive green slopes under bright, blue summer skies. The narrative highlights the harmony and tranquility of the setting, emphasizing the natural beauty and peaceful atmosphere. Character interactions are portrayed with nuance: figures like Mr. Boythorn exhibit exuberant, protective personalities; conversations among characters such as Mr. Skimpole and others explore themes of earnestness, social behavior, and personal philosophy, often contrasting sincere effort with complacency. 

The text also touches upon social restrictions, mysterious relationships—such as a woman marrying a figure-head baronet—and subtle character nuances, including unspoken feelings and social dynamics. Descriptions of stormy weather evoke awe and reflect on the power and beneficenc

## Section 4 — Query the chapter

Use RAPTOR for open-ended questions: *"What does this chapter say about X?"*  
The collapsed tree retrieval finds the most relevant nodes regardless of where in the chapter they appear.

In [6]:
importlib.reload(booksum_pipeline)
from booksum_pipeline import query_chapter

question = "What does this chapter say about the relationship between the characters?"

answer = query_chapter(row, question=question, config=config)
print(f'Q: {question}')
print(f'\nA: {answer}')

2026-04-07 14:13:01,798 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
            Selection Mode: top_k
            Summarization Length: 100
            Summarization Model: <raptor.SummarizationModels.GPT4NanoSummarizationModel object at 0x00000190AB07BCB0>
            Embedding Models: {'EMB': <raptor.EmbeddingModels.SBertEmbeddingModel object at 0x00000190AB07AA50>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        
2026-04-07 14:13:01,799 - Successfully initialized ClusterTreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
           

Q: What does this chapter say about the relationship between the characters?

A: This chapter portrays a complex web of relationships among the characters, emphasizing themes of social indifference, personal pride, eccentricity, and emotional undercurrents. The interactions suggest that many characters maintain a certain distance or aloofness from one another, often shaped by societal expectations or personal dispositions. For example, Lady Dedlock's admiration for the pretty girl indicates her fondness and favoritism, yet her dismissive reaction toward the narrator hints at a reserved or guarded nature. The narrator’s emotional reaction upon making eye contact with Lady Dedlock reveals a deeper, perhaps unspoken connection or buried history, suggesting that their relationship is emotionally charged and past-oriented.

Characters such as Mr. Boythorn display a humorous self-awareness and eccentricity, confidently asserting his own importance and teasing social hierarchies, which indica

## Section 5 — Evaluate summary

In [7]:
importlib.reload(qasper_pipeline)
from qasper_pipeline import print_evaluation

scores = print_evaluation(result['summary'], reference)

REFERENCE:
In Chapter 18, Richard begins at Kenge and Carboy's. The rest of the party leave him in London, and go to visit Mr. Boythorn at his place down in Lincolnshire, which neighbors the Dedlocks' estate. When the party goes to church, Esther sees not only Rosa, Mrs. Rouncewell, and Mademoiselle Hortense, but also Sir Leicester and Lady Dedlock. Esther seems to recognize Lady Dedlock's face, and wonders at the resemblance to her godmother's face. She also vaguely realizes that Lady Dedlock's face resembles her own when seen in the mirror. Later, Esther, Ada, and John Jarndyce then go for a walk and take refuge in a gamekeeper's lodge when it rains. Lady Dedlock is already there, and Esther experiences another weird feeling upon meeting Lady Dedlock. Lady Dedlock and John Jarndyce are old acquaintances, but it appears Mr. Jarndyce was actually closer to Lady Dedlock's sister. They discuss Lady Dedlock's sister's death, as she "died in retirement. Esther admires Lady Dedlock's beauty

2026-04-07 14:14:15,280 - Using default tokenizer.
2026-04-07 14:14:15,479 - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-07 14:14:15,619 - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-07 14:14:15,787 - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-07 14:14:15,913 - HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-07 14:14:16,028 - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-07 14:14:16,137 - HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main?recursive=true&expand=false 

SCORES:
  ROUGE-1  : 0.1814
  ROUGE-2  : 0.0140
  ROUGE-L  : 0.1070
  BERTScore: P=0.8203  R=0.8037  F1=0.8119


## Section 6 — Benchmark: RAPTOR vs Baseline

Compare RAPTOR against a plain LLM call (same model, same prompt, raw truncated text).

In [8]:
importlib.reload(booksum_pipeline)
from booksum_pipeline import compare_chapter

results = compare_chapter(row, config=config)

Document: Bleak House.chapter 18
Running baseline...


2026-04-07 14:14:30,002 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-07 14:14:30,011 - Using default tokenizer.
2026-04-07 14:14:30,297 - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-07 14:14:30,433 - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-07 14:14:30,559 - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-07 14:14:30,742 - HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-07 14:14:30,870 - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-07 14:14:30,987 - HTTP Requ

Running RAPTOR...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]









































































































Batches: 100%|██████████| 1/1 [00:02<00:00,  2.32s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:02<00:00,  2.50s/it]







Batches: 100%|██████████| 1/1 [00:02<00:00,  2.46s/it]





Batches: 100%|██████████| 1/1 [00:02<00:00,  2.51s/it]








Batches: 100%|██████████| 1/1 [00:02<00:00,  2.56s/it]




Batches: 100%|██████████| 1/1 [00:02<00:00,  2.57s/it]























Batches: 100%|██████████| 1/1 [00:02<00:00,  2.55s/it]








Batches: 100%|██████████| 1/1 [00:02<00:00,  2.70s/it]












Batches: 100%|██████████| 1/1 [00:02<00:00,  2.66s/it]













Batches: 100%|██████████| 1/1 [00:02<00:00,  2.68s/it]

















Batches: 100%|██████████| 1/1 [00:02<00:00,  2.84s/it]










Batches: 100%|██████████| 1/1 [00:02<00:00,  2.81s/it]



































BASELINE SUMMARY (plain LLM, no RAPTOR):
--------------------------------------------------------------------------------
The excerpt narrates the ongoing developments in Lady Dedlock's circle, focusing on Richard's indecisiveness about his career path between law and medicine, his reluctant experimentation with Mr. Kenge's legal practice, and his cheerful, reckless personality. Richard's financial dealings are characterized by generosity and wastefulness, as he makes haphazard purchases and boasts about saving money through seemingly trivial transactions. Meanwhile, his guardians discuss arrangements for his residence in London to support his legal studies, settling on a discreet, furnished lodging near Queen Square.

The journey continues with visits to friends, including Mr. Skimpole, whose humorous aversion to money and penchant for borrowing are highlighted, alongside anecdotes about his amusing yet impractical attitudes toward expenses and security. The travel scene captures the